<a href="https://colab.research.google.com/github/RedRangerWentWild/Finzo/blob/main/tax_rag_month2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tax RAG System — Month 2
Answers tax questions by retrieving relevant sections from the Income Tax Act 2025.



In [1]:
# Cell 1 — Install dependencies (run once, ~3 min)
!pip install -q chromadb sentence-transformers pdfplumber langchain langchain-community
!pip install -q transformers accelerate bitsandbytes
print('✅ Dependencies installed')

✅ Dependencies installed


In [2]:
# Cell 2 — Build a synthetic knowledge base
# In production: replace this with the real IT Act 2025 PDF
# Download from: https://incometaxindia.gov.in

TAX_KNOWLEDGE = [
    {
        "section": "Section 80C",
        "text": "Section 80C allows deduction up to Rs 1,50,000 for investments in PPF, ELSS, NSC, life insurance premiums, EPF contributions, home loan principal repayment, tuition fees for children, tax-saving FDs of 5 years, and NPS contributions. Available only under old tax regime."
    },
    {
        "section": "Section 80D",
        "text": "Section 80D provides deduction for health insurance premiums. Individuals can claim up to Rs 25,000 for self and family, plus Rs 25,000 for parents. If parents are senior citizens, the limit increases to Rs 50,000. Total maximum deduction is Rs 75,000."
    },
    {
        "section": "HRA Exemption",
        "text": "HRA exemption is the minimum of: (1) actual HRA received, (2) 50% of basic salary for metro cities or 40% for non-metro, (3) rent paid minus 10% of basic salary. Metro cities are Mumbai, Delhi, Chennai, Kolkata. Only available under old tax regime."
    },
    {
        "section": "New Tax Regime 2025",
        "text": "Under the new tax regime for FY 2025-26: income up to Rs 3 lakh is nil, Rs 3-7 lakh taxed at 5%, Rs 7-10 lakh at 10%, Rs 10-12 lakh at 15%, Rs 12-15 lakh at 20%, above Rs 15 lakh at 30%. Standard deduction of Rs 75,000 is allowed. Rebate u/s 87A available for income up to Rs 12 lakh making effective tax nil."
    },
    {
        "section": "Old Tax Regime",
        "text": "Under the old tax regime: income up to Rs 2.5 lakh is nil, Rs 2.5-5 lakh at 5%, Rs 5-10 lakh at 20%, above Rs 10 lakh at 30%. Allows all deductions under Chapter VI-A including 80C, 80D, HRA, LTA, home loan interest under Section 24b. Surcharge applies on income above Rs 50 lakh."
    },
    {
        "section": "Section 24b Home Loan",
        "text": "Section 24b allows deduction of interest on home loan up to Rs 2,00,000 per year for self-occupied property. For let-out property, the entire interest is deductible. The property must be acquired or constructed within 5 years from the end of financial year in which loan was taken."
    },
    {
        "section": "ITR Form Selection",
        "text": "ITR-1 (Sahaj): Salaried individuals with income up to Rs 50 lakh, one house property, other sources. ITR-2: Capital gains, more than one property, foreign income, income above Rs 50 lakh. ITR-3: Business or profession income (non-presumptive). ITR-4 (Sugam): Presumptive income under sections 44AD, 44ADA, 44AE."
    },
    {
        "section": "Section 80CCD NPS",
        "text": "Section 80CCD(1B) provides additional deduction of Rs 50,000 for NPS contributions over and above the Rs 1.5 lakh 80C limit. This means total deduction for NPS can be Rs 2 lakh. Section 80CCD(2) allows employer NPS contribution deduction up to 14% of salary for government employees and 10% for others."
    },
    {
        "section": "TDS and Form 26AS",
        "text": "Form 26AS is a consolidated tax statement showing all TDS deducted by employers and others, advance tax paid, self-assessment tax paid, and refunds. It is available on the income tax portal. Taxpayers must reconcile Form 26AS with Form 16 before filing ITR. Discrepancies can lead to notices from the income tax department."
    },
    {
        "section": "Capital Gains",
        "text": "Short-term capital gains on equity (held less than 12 months) are taxed at 20% from FY 2024-25. Long-term capital gains on equity above Rs 1.25 lakh are taxed at 12.5% without indexation. For debt funds and property, long-term is 24 months, taxed at 20% with indexation or 12.5% without."
    },
    {
        "section": "Rebate 87A",
        "text": "Section 87A provides a tax rebate for resident individuals. Under new regime, if total income does not exceed Rs 12 lakh, tax liability is nil after rebate. Under old regime, rebate of Rs 12,500 is available if income does not exceed Rs 5 lakh. The rebate equals the tax payable or the limit, whichever is lower."
    },
    {
        "section": "Filing Deadlines",
        "text": "The due date for filing ITR for individuals not requiring audit is July 31 of the assessment year. For those requiring audit, it is October 31. Belated returns can be filed until December 31 with a late fee of Rs 5,000 (Rs 1,000 if income is below Rs 5 lakh). Revised returns can be filed until December 31."
    }
]

print(f'✅ Knowledge base created with {len(TAX_KNOWLEDGE)} sections')

✅ Knowledge base created with 12 sections


In [3]:
# Cell 3 — Create embeddings and build ChromaDB vector store
# Fix for opentelemetry dependency conflict
!pip install --upgrade opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp-proto-grpc opentelemetry-exporter-otlp-proto-http opentelemetry-proto
# Temporarily uninstall and reinstall chromadb to ensure it picks up the new opentelemetry versions
# This is often needed in Colab to resolve persistent dependency issues
!pip uninstall -y chromadb
!pip install chromadb

import chromadb
from sentence_transformers import SentenceTransformer

print('Loading embedding model (all-MiniLM-L6-v2)...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')  # 80MB, fast on CPU

# Create ChromaDB (in-memory for Colab)
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name='tax_knowledge',
    metadata={'hnsw:space': 'cosine'}
)

# Embed and store all chunks
texts = [item['text'] for item in TAX_KNOWLEDGE]
ids   = [f'chunk_{i}' for i in range(len(TAX_KNOWLEDGE))]
metas = [{'section': item['section']} for item in TAX_KNOWLEDGE]

print('Embedding knowledge base...')
embeddings = embedder.encode(texts, show_progress_bar=True).tolist()

collection.add(documents=texts, embeddings=embeddings, ids=ids, metadatas=metas)
print(f'\n✅ Vector store built — {collection.count()} chunks indexed')

Found existing installation: chromadb 1.5.8
Uninstalling chromadb-1.5.8:
  Successfully uninstalled chromadb-1.5.8
  Using cached chromadb-1.5.8-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
Using cached chromadb-1.5.8-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (23.2 MB)
Loading embedding model (all-MiniLM-L6-v2)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding knowledge base...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Vector store built — 12 chunks indexed


In [4]:
print('Installing/Upgrading bitsandbytes...')
!pip install -U bitsandbytes>=0.46.1
print('✅ bitsandbytes installed')

Installing/Upgrading bitsandbytes...
✅ bitsandbytes installed


In [5]:
# Cell 4 — Test retrieval (no LLM needed, just checks similarity search)
def retrieve(question, top_k=3):
    q_embedding = embedder.encode([question]).tolist()
    results = collection.query(
        query_embeddings=q_embedding,
        n_results=top_k
    )
    chunks = []
    for i in range(len(results['documents'][0])):
        chunks.append({
            'section': results['metadatas'][0][i]['section'],
            'text': results['documents'][0][i],
            'distance': results['distances'][0][i]
        })
    return chunks

# Test it
test_q = 'Can I claim HRA if I live in a rented apartment in Delhi?'
retrieved = retrieve(test_q)

print(f'Question: {test_q}\n')
print('Top retrieved chunks:')
for i, chunk in enumerate(retrieved):
    print(f'\n[{i+1}] {chunk["section"]} (distance: {chunk["distance"]:.3f})')
    print(f'    {chunk["text"][:120]}...')

Question: Can I claim HRA if I live in a rented apartment in Delhi?

Top retrieved chunks:

[1] HRA Exemption (distance: 0.379)
    HRA exemption is the minimum of: (1) actual HRA received, (2) 50% of basic salary for metro cities or 40% for non-metro,...

[2] Section 80D (distance: 0.668)
    Section 80D provides deduction for health insurance premiums. Individuals can claim up to Rs 25,000 for self and family,...

[3] Rebate 87A (distance: 0.685)
    Section 87A provides a tax rebate for resident individuals. Under new regime, if total income does not exceed Rs 12 lakh...


In [6]:
# Cell 5 — Rule-based answer engine (no LLM tokens needed, zero cost)
# Uses retrieved context + simple prompt template
# For demo: uses the rule engine from Month 1 + retrieved context
# For production: plug in Mistral-7B below

def build_prompt(question, retrieved_chunks):
    context = '\n\n'.join([
        f"[{c['section']}]\n{c['text']}"
        for c in retrieved_chunks
    ])
    return f"""You are an expert Indian tax advisor. Answer the question using ONLY the context below.
If the context doesn't contain enough information, say so clearly. Never guess.

CONTEXT:
{context}

QUESTION: {question}

ANSWER (be specific, mention exact amounts/limits, cite the section):"""

# Demo with rule-based response (shows the retrieved context nicely)
def rag_answer_demo(question):
    chunks = retrieve(question, top_k=3)
    prompt = build_prompt(question, chunks)

    print('=' * 60)
    print(f'Q: {question}')
    print('=' * 60)
    print('\nRetrieved sections:')
    for c in chunks:
        print(f'  - {c["section"]}')
    print('\nPrompt being sent to LLM:')
    print('-' * 40)
    print(prompt[:500] + '...')
    print('=' * 60)
    return prompt

rag_answer_demo('What is the maximum I can save under 80C?')

Q: What is the maximum I can save under 80C?

Retrieved sections:
  - Section 80C
  - Section 80D
  - Section 80CCD NPS

Prompt being sent to LLM:
----------------------------------------
You are an expert Indian tax advisor. Answer the question using ONLY the context below.
If the context doesn't contain enough information, say so clearly. Never guess.

CONTEXT:
[Section 80C]
Section 80C allows deduction up to Rs 1,50,000 for investments in PPF, ELSS, NSC, life insurance premiums, EPF contributions, home loan principal repayment, tuition fees for children, tax-saving FDs of 5 years, and NPS contributions. Available only under old tax regime.

[Section 80D]
Section 80D provides d...


"You are an expert Indian tax advisor. Answer the question using ONLY the context below.\nIf the context doesn't contain enough information, say so clearly. Never guess.\n\nCONTEXT:\n[Section 80C]\nSection 80C allows deduction up to Rs 1,50,000 for investments in PPF, ELSS, NSC, life insurance premiums, EPF contributions, home loan principal repayment, tuition fees for children, tax-saving FDs of 5 years, and NPS contributions. Available only under old tax regime.\n\n[Section 80D]\nSection 80D provides deduction for health insurance premiums. Individuals can claim up to Rs 25,000 for self and family, plus Rs 25,000 for parents. If parents are senior citizens, the limit increases to Rs 50,000. Total maximum deduction is Rs 75,000.\n\n[Section 80CCD NPS]\nSection 80CCD(1B) provides additional deduction of Rs 50,000 for NPS contributions over and above the Rs 1.5 lakh 80C limit. This means total deduction for NPS can be Rs 2 lakh. Section 80CCD(2) allows employer NPS contribution deductio

In [7]:
# Cell 6 — OPTIONAL: Connect real LLM (uses GPU credits, skip for demo)
# Uncomment only if you want actual generated answers
# Switch runtime to T4 GPU before running this cell

USE_REAL_LLM = True  # Set True to enable

if USE_REAL_LLM:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    import torch

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16
    )

    print('Loading Mistral-7B in 4-bit (needs ~5GB VRAM)...')
    tokenizer = AutoTokenizer.from_pretrained('mistralai/Mistral-7B-Instruct-v0.2')
    model = AutoModelForCausalLM.from_pretrained(
        'mistralai/Mistral-7B-Instruct-v0.2',
        quantization_config=bnb_config,
        device_map='auto'
    )
    print('✅ Model loaded')

    def rag_answer_llm(question):
        chunks = retrieve(question, top_k=3)
        prompt = build_prompt(question, chunks)
        inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=300, temperature=0.1)
        answer = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return answer

    print(rag_answer_llm('Can I claim both 80C and NPS deduction?'))
else:
    print('LLM disabled. Set USE_REAL_LLM = True and switch to GPU runtime to enable.')

Loading Mistral-7B in 4-bit (needs ~5GB VRAM)...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


✅ Model loaded


Yes, you can claim deductions under Sections 80C and 80CCD(1B) for investments in PPF, ELSS, NSC, life insurance premiums, EPF contributions, home loan principal repayment, tuition fees for children, tax-saving FDs of 5 years, and NPS contributions up to the limit of Rs 1,50,000 under Section 80C. Additionally, you can claim an extra deduction of up to Rs 50,000 under Section 80CCD(1B) for NPS contributions. The total deduction for NPS can be up to Rs 2 lakh (Rs 1.5 lakh under 80C + Rs 50,000 under 80CCD(1B)).


In [2]:
# Cell 7 — Interactive demo widget (works on CPU, great for prof demo)
import ipywidgets as widgets
from IPython.display import display, clear_output

SAMPLE_QUESTIONS = [
    'Can I claim HRA if I pay rent to my parents?',
    'Which ITR form should a salaried person with mutual funds use?',
    'What is the tax on long-term capital gains from stocks?',
    'How much can I save under 80C and NPS combined?',
    'What is the deadline to file my ITR?',
    'Should I choose old or new tax regime?'
]

dropdown = widgets.Dropdown(
    options=['(type your own question below)'] + SAMPLE_QUESTIONS,
    description='Sample Q:',
    layout=widgets.Layout(width='80%')
)
text_input = widgets.Text(
    placeholder='Or type your tax question here...',
    layout=widgets.Layout(width='80%')
)
btn = widgets.Button(description='Ask', button_style='success')
out = widgets.Output()

def on_ask(b):
    question = text_input.value.strip()
    if not question and dropdown.value != '(type your own question below)':
        question = dropdown.value
    if not question:
        return
    with out:
        clear_output()
        chunks = retrieve(question, top_k=3)
        print('=' * 65)
        print(f' Q: {question}')
        print('=' * 65)
        print('\n Retrieved from IT Act knowledge base:')
        for i, c in enumerate(chunks, 1):
            print(f'\n  [{i}] {c["section"]}')
            print(f'      {c["text"][:200]}...')
        print('\n' + '=' * 65)
        print(' In production: Mistral-7B would generate a final answer')
        print(' using the above sections as grounded context.')
        print('=' * 65)

btn.on_click(on_ask)
display(widgets.VBox([dropdown, text_input, btn, out]))

In [ ]:
# Cell 8 — Save knowledge base for reuse (so you don't re-embed next time)
import pickle

save_data = {
    'knowledge': TAX_KNOWLEDGE,
    'embeddings': embeddings,
    'ids': ids
}

with open('tax_kb.pkl', 'wb') as f:
    pickle.dump(save_data, f)

from google.colab import files
files.download('tax_kb.pkl')
print('✅ Knowledge base saved and downloaded')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Knowledge base saved and downloaded


## When to add the real IT Act PDF

Replace Cell 2 with:

```python
import pdfplumber
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Upload PDF first
from google.colab import files
uploaded = files.upload()  # upload income_tax_act_2025.pdf

# Extract text
with pdfplumber.open('income_tax_act_2025.pdf') as pdf:
    full_text = '\n'.join(page.extract_text() or '' for page in pdf.pages)

# Chunk it
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(full_text)
print(f'PDF split into {len(chunks)} chunks')
```

Then continue from Cell 3 with the real chunks.